In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import networkx as nx
from shapely.geometry import LineString, MultiLineString

In [8]:
# =========================================================
# 1. PARAMÈTRES
# =========================================================

# Remplace ce chemin par le bon chemin vers ton shapefile
SHP_PATH = "ROUTE500_3-0__SHP_LAMB93_FXX_2021-11-03/ROUTE500/1_DONNEES_LIVRAISON_2022-01-00175/R500_3-0_SHP_LAMB93_FXX-ED211/RESEAU_ROUTIER/TRONCON_ROUTE.shp"

# =========================================================
# 2. CHARGEMENT DES DONNÉES
# =========================================================

gdf = gpd.read_file(SHP_PATH, encoding="latin1")

print("Colonnes disponibles :")
print(gdf.columns.tolist())
print("\nCRS initial :", gdf.crs)
print("Nombre de lignes :", len(gdf))

# Passage en Lambert-93 si besoin pour avoir des longueurs en mètres
if gdf.crs is None or gdf.crs.to_epsg() != 2154:
    gdf = gdf.to_crs(2154)

print("CRS après reprojection :", gdf.crs)

# =========================================================
# 3. NETTOYAGE + LONGUEURS
# =========================================================

# Supprimer géométries nulles
gdf = gdf[gdf.geometry.notna()].copy()

# Garder seulement lignes / multilignes
gdf = gdf[gdf.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()

# Longueur en mètres
gdf["length_m"] = gdf.geometry.length

# Supprimer segments trop courts ou vides
gdf = gdf[gdf["length_m"] > 0].copy()

# =========================================================
# 4. CONSTRUCTION ROBUSTE DE LA VITESSE
# =========================================================

def build_speed_column(df: pd.DataFrame) -> pd.Series:
    """
    Construit une colonne speed_kmh robuste.
    Priorité :
    1) VIT_MOY_VL si présente
    2) IMPORTANCE si présente
    3) NATURE si présente
    4) vitesse par défaut
    """

    if "VIT_MOY_VL" in df.columns:
        speed = pd.to_numeric(df["VIT_MOY_VL"], errors="coerce")
        return speed.fillna(80)

    if "IMPORTANCE" in df.columns:
        # mapping simple et défendable pour un MVP
        speed_map = {
            "1": 110,
            "2": 100,
            "3": 90,
            "4": 80,
            "5": 70,
            "6": 50
        }
        speed = df["IMPORTANCE"].astype(str).map(speed_map)
        return speed.fillna(80)

    if "NATURE" in df.columns:
        speed_map = {
            "Autoroute": 130,
            "Quasi-autoroute": 110,
            "Bretelle": 50,
            "Route à 2 chaussées": 90,
            "Route à 1 chaussée": 80,
            "Rond-point": 30,
            "Type autoroutier": 110
        }
        speed = df["NATURE"].map(speed_map)
        return speed.fillna(80)

    return pd.Series(80, index=df.index)

gdf["speed_kmh"] = build_speed_column(gdf)

# Sécurité
gdf["speed_kmh"] = pd.to_numeric(gdf["speed_kmh"], errors="coerce").fillna(80)
gdf.loc[gdf["speed_kmh"] <= 0, "speed_kmh"] = 80

# Temps de trajet estimé en secondes
gdf["travel_time_s"] = gdf["length_m"] / (gdf["speed_kmh"] * 1000 / 3600)

print("\nExemple après création des vitesses :")
print(gdf[["length_m", "speed_kmh", "travel_time_s"]].head())

# =========================================================
# 5. EXTRACTION DES EXTRÉMITÉS DES TRONÇONS
# =========================================================

def get_endpoints(geom):
    """
    Retourne (u, v) = coordonnées début et fin du tronçon
    sous forme de tuples (x, y).
    """
    if geom is None:
        return None, None

    if isinstance(geom, LineString):
        coords = list(geom.coords)
        if len(coords) < 2:
            return None, None
        return tuple(coords[0]), tuple(coords[-1])

    if isinstance(geom, MultiLineString):
        parts = list(geom.geoms)
        if len(parts) == 0:
            return None, None

        first_coords = list(parts[0].coords)
        last_coords = list(parts[-1].coords)

        if len(first_coords) == 0 or len(last_coords) == 0:
            return None, None

        return tuple(first_coords[0]), tuple(last_coords[-1])

    return None, None

gdf[["u", "v"]] = gdf.apply(
    lambda row: pd.Series(get_endpoints(row.geometry)),
    axis=1
)

gdf = gdf[gdf["u"].notna() & gdf["v"].notna()].copy()

print("\nNombre de tronçons exploitables :", len(gdf))

# =========================================================
# 6. CONSTRUCTION DU GRAPHE ROUTIER
# =========================================================

G = nx.DiGraph()

def add_edge_with_direction(graph, row):
    u = row["u"]
    v = row["v"]

    attrs = {
        "length_m": float(row["length_m"]),
        "travel_time_s": float(row["travel_time_s"]),
        "speed_kmh": float(row["speed_kmh"]),
        "geometry": row["geometry"]
    }

    # Ajouter d'autres attributs si présents
    optional_cols = ["NATURE", "IMPORTANCE", "NB_VOIES", "SENS", "NUMERO"]
    for col in optional_cols:
        if col in row.index:
            attrs[col] = row[col]

    sens = row["SENS"] if "SENS" in row.index else "Double sens"

    if pd.isna(sens):
        sens = "Double sens"

    sens_str = str(sens).strip().lower()

    # Gestion simple et robuste des sens
    if "inverse" in sens_str:
        graph.add_edge(v, u, **attrs)

    elif "direct" in sens_str:
        graph.add_edge(u, v, **attrs)

    else:
        # par défaut : double sens
        graph.add_edge(u, v, **attrs)
        graph.add_edge(v, u, **attrs)

for _, row in gdf.iterrows():
    add_edge_with_direction(G, row)

print("\nGraphe construit :")
print("Nombre de nœuds :", G.number_of_nodes())
print("Nombre d'arcs :", G.number_of_edges())

# =========================================================
# 7. FONCTION POUR TROUVER LE NŒUD LE PLUS PROCHE
# =========================================================

# Pour un premier prototype, on fait une recherche simple.
# Ce n'est pas la plus rapide sur gros volume, mais c'est simple et fonctionne.

nodes_list = list(G.nodes())
nodes_array = np.array(nodes_list)  # shape = (n, 2)

def nearest_node(x, y):
    diffs = nodes_array - np.array([x, y])
    dists = np.sqrt((diffs ** 2).sum(axis=1))
    idx = np.argmin(dists)
    return tuple(nodes_array[idx])

# =========================================================
# 8. EXEMPLE DE ROUTAGE
# =========================================================

# Exemple : on prend deux tronçons aléatoires pour choisir un départ/arrivée
sample = gdf.sample(2, random_state=42).copy()

origin_geom = sample.iloc[0].geometry
dest_geom = sample.iloc[1].geometry

origin_point = origin_geom.interpolate(0.5, normalized=True)
dest_point = dest_geom.interpolate(0.5, normalized=True)

origin_node = nearest_node(origin_point.x, origin_point.y)
dest_node = nearest_node(dest_point.x, dest_point.y)

print("\nOrigine :", origin_node)
print("Destination :", dest_node)

# Calcul du plus court chemin en temps
try:
    path = nx.shortest_path(G, source=origin_node, target=dest_node, weight="travel_time_s")
    travel_time_total_s = nx.shortest_path_length(
        G,
        source=origin_node,
        target=dest_node,
        weight="travel_time_s"
    )

    # Calcul distance totale
    total_length_m = 0
    for i in range(len(path) - 1):
        total_length_m += G[path[i]][path[i + 1]]["length_m"]

    print("\nChemin trouvé")
    print("Nombre de nœuds dans le chemin :", len(path))
    print("Distance totale (km) :", round(total_length_m / 1000, 2))
    print("Temps estimé (min) :", round(travel_time_total_s / 60, 2))

except nx.NetworkXNoPath:
    print("\nAucun chemin trouvé entre ces deux nœuds.")
except Exception as e:
    print("\nErreur pendant le calcul du chemin :", e)

# =========================================================
# 9. EXPORT OPTIONNEL DU GRAPHE D'ARÊTES
# =========================================================

# Si tu veux garder une table exploitable
edge_cols = ["u", "v", "length_m", "speed_kmh", "travel_time_s"]
extra_cols = [c for c in ["NATURE", "IMPORTANCE", "NB_VOIES", "SENS", "NUMERO"] if c in gdf.columns]
export_cols = edge_cols + extra_cols + ["geometry"]

gdf_export = gdf[export_cols].copy()

print("\nTable finale prête pour analyses :")
print(gdf_export.head())

Colonnes disponibles :
['ID_RTE500', 'VOCATION', 'NB_CHAUSSE', 'NB_VOIES', 'ETAT', 'ACCES', 'RES_VERT', 'SENS', 'NUM_ROUTE', 'RES_EUROPE', 'LONGUEUR', 'CLASS_ADM', 'geometry']

CRS initial : PROJCS["RGF93 Lambert 93",GEOGCS["RGF93 geographiques (dms)",DATUM["Reseau_Geodesique_Francais_1993_v1",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6171"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["IGNF","RGF93G"]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",46.5],PARAMETER["central_meridian",3],PARAMETER["standard_parallel_1",44],PARAMETER["standard_parallel_2",49],PARAMETER["false_easting",700000],PARAMETER["false_northing",6600000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["IGNF","LAMB93"]]
Nombre de lignes : 1302758
CRS après reprojection : PROJCS["RGF93 Lambert 93",GEOGCS["RGF93 geographiques (dms)

In [9]:
print(gdf.columns.tolist())

['ID_RTE500', 'VOCATION', 'NB_CHAUSSE', 'NB_VOIES', 'ETAT', 'ACCES', 'RES_VERT', 'SENS', 'NUM_ROUTE', 'RES_EUROPE', 'LONGUEUR', 'CLASS_ADM', 'geometry', 'length_m', 'speed_kmh', 'travel_time_s', 'u', 'v']


In [10]:
print(gdf_export.head(10).to_string())

                       u                      v     length_m  speed_kmh  travel_time_s    NB_VOIES         SENS                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 geometry
0  (894833.5, 6265743.5)  (894051.2, 6262805.5)  3184.817366         80     143.316781  Sans objet  Double sens                                                                                                                    LINESTRING (894833.5 6265743.5, 894824.7 6265683.4, 894809.2 6265628.6, 8947